# Agent Tools

**Module:** 10-agentic-ai-concepts

**Notebook:** `06-agent-tools.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Tool Interface Design** with clear contracts and failure modes
- Explain and apply **Tool Categories** with clear contracts and failure modes
- Explain and apply **Observation Contracts** with clear contracts and failure modes
- Explain and apply **Argument Validation** with clear contracts and failure modes
- Explain and apply **Permissions** with clear contracts and failure modes
- Explain and apply **Toolsets for Common Agents** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Agent Tools

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Tool Interface Design**
2. **Tool Categories**
3. **Observation Contracts**
4. **Argument Validation**
5. **Permissions**
6. **Toolsets for Common Agents**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Tool Interface Design

### Definition
**Tool Interface Design** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Tool Interface Design typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tool Interface Design: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Tool Interface Design as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tool Interface Design as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tool Interface Design
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Tool Interface Design when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Tool Interface Design improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Tool Interface Design" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tool Interface Design"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Tool Categories

### Definition
**Tool Categories** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Tool Categories typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Tool Categories: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Tool Categories as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Tool Categories as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Tool Categories
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Tool Categories when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Tool Categories" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Tool Categories"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Tool Categories

**Situation:** A team wants to productionize a feature involving **Tool Categories**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Observation Contracts

### Definition
**Observation Contracts** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Observation Contracts typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Observation Contracts: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Observation Contracts as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Observation Contracts as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Observation Contracts
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Observation Contracts when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Observation Contracts" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Observation Contracts"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Observation Contracts"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Observation Contracts"}
strong = {"definition": "Observation Contracts", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Observation Contracts"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Observation Contracts", "passed": len(checks)-len(failed), "failed": failed})


## Argument Validation

### Definition
**Argument Validation** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Argument Validation typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Argument Validation: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Argument Validation as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Argument Validation as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Argument Validation
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Argument Validation when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Argument Validation" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Argument Validation"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


### Worked scenario — Argument Validation

**Situation:** A team wants to productionize a feature involving **Argument Validation**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Permissions

### Definition
**Permissions** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Permissions typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Permissions: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Permissions as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Permissions as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Permissions
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Permissions when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Permissions" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Permissions"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Permissions"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Permissions"}
strong = {"definition": "Permissions", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Permissions"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Permissions", "passed": len(checks)-len(failed), "failed": failed})


## Toolsets for Common Agents

### Definition
**Toolsets for Common Agents** is a core building block in 06-agent-tools within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Toolsets for Common Agents typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Toolsets for Common Agents: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Toolsets for Common Agents as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Toolsets for Common Agents as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Toolsets for Common Agents
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Toolsets for Common Agents when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Toolsets for Common Agents" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Toolsets for Common Agents"
    notebook: str = "06-agent-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Toolsets for Common Agents

**Situation:** A team wants to productionize a feature involving **Toolsets for Common Agents**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Agent Tools**.

| Topic | Do | Don't |
|-------|----|-------|
| Tool Interface Design | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Tool Categories | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Observation Contracts | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Argument Validation | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Permissions | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Toolsets for Common Agents | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Tool Interface Design | Key concept covered in this notebook; see its section for definition and pitfalls |
| Tool Categories | Key concept covered in this notebook; see its section for definition and pitfalls |
| Observation Contracts | Key concept covered in this notebook; see its section for definition and pitfalls |
| Argument Validation | Key concept covered in this notebook; see its section for definition and pitfalls |
| Permissions | Key concept covered in this notebook; see its section for definition and pitfalls |
| Toolsets for Common Agents | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Agent Tools** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **10-agentic-ai-concepts**.


## Try It Yourself

1. Implement a failing test/fixture for **Tool Interface Design**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Tool Categories**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Observation Contracts**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Argument Validation**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Permissions**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
